# Домашнее задание 7.1: Остывание стержня — можно ли доверять устойчивой схеме?

Рассматривается одномерное уравнение теплопроводности на отрезке $x \in [0, 1]$ с нулевыми граничными условиями:
$$u_t = u_{xx}, \quad x \in [0, 1], \quad u(0, t) = u(1, t) = 0$$

Начальное распределение в первой части:
$$u(x, 0) = \sin(\pi x) + 0.3\sin(5\pi x)$$

Точное решение:
$$u(x, t) = e^{-\pi^2 t}\sin(\pi x) + 0.3e^{-25\pi^2 t}\sin(5\pi x)$$

---

## Часть 1. Явная схема FTCS и неявная схема Эйлера

### Шаг 1. Разностные схемы и метод прогонки

Вводим равномерную сетку $x_j = j\Delta x$ ($\Delta x = 1/N$) и шаг по времени $\Delta t = r\Delta x^2$ при $r = 0.4$.


Трёхдиагональная система на каждом шаге решается методом прогонки.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Точное решение задачи
def exact_solution(x, t):
    return np.exp(-np.pi**2 * t) * np.sin(np.pi * x) + 0.3 * np.exp(-25 * np.pi**2 * t) * np.sin(5 * np.pi * x)

# Начальное условие первой части
def u0_smooth(x):
    return np.sin(np.pi * x) + 0.3 * np.sin(5 * np.pi * x)

# Метод прогонки для трехдиагональной СЛАУ
def thomas_solve(a, b, c, d):
    m = len(d)
    c_star = np.zeros(m)
    d_star = np.zeros(m)
    x = np.zeros(m)
    
    c_star[0] = c[0] / b[0]
    d_star[0] = d[0] / b[0]
    for i in range(1, m):
        denom = b[i] - a[i] * c_star[i - 1]
        if i < m - 1:
            c_star[i] = c[i] / denom
        d_star[i] = (d[i] - a[i] * d_star[i - 1]) / denom
        
    x[-1] = d_star[-1]
    for i in range(m - 2, -1, -1):
        x[i] = d_star[i] - c_star[i] * x[i + 1]
    return x

# Расчет по явной схеме FTCS до момента t_end
def solve_ftcs(N, t_end, r=0.4):
    dx = 1.0 / N
    dt = r * dx**2
    x = np.linspace(0.0, 1.0, N + 1)
    u = u0_smooth(x)
    t = 0.0
    while t < t_end - 1e-12:
        dt_step = min(dt, t_end - t)
        r_step = dt_step / dx**2
        u_next = np.zeros_like(u)
        for j in range(1, N):
            u_next[j] = u[j] + r_step * (u[j - 1] - 2.0 * u[j] + u[j + 1])
        u = u_next
        t += dt_step
    return x, u

# Расчет по неявной схеме Эйлера до момента t_end
def solve_implicit_euler(N, t_end, r=0.4):
    dx = 1.0 / N
    dt = r * dx**2
    x = np.linspace(0.0, 1.0, N + 1)
    u = u0_smooth(x)
    t = 0.0
    m = N - 1
    while t < t_end - 1e-12:
        dt_step = min(dt, t_end - t)
        r_step = dt_step / dx**2
        a = np.full(m, -r_step)
        b = np.full(m, 1.0 + 2.0 * r_step)
        c = np.full(m, -r_step)
        d = u[1:-1].copy()
        u[1:-1] = thomas_solve(a, b, c, d)
        u[0] = 0.0
        u[-1] = 0.0
        t += dt_step
    return x, u

### Шаг 2. Профили температуры при $t = 0.001, 0.01, 0.1$

Построим графики численных и точного решений в указанные моменты времени на сетке $N = 50$ при $r = 0.4$.

In [ ]:
t_list = [0.001, 0.01, 0.1]
N_plot = 50
x_fine = np.linspace(0.0, 1.0, 400)

for t_val in t_list:
    x_num, u_ftcs = solve_ftcs(N_plot, t_val, r=0.4)
    _, u_ie = solve_implicit_euler(N_plot, t_val, r=0.4)
    
    plt.figure(figsize=(7, 5))
    plt.plot(x_fine, exact_solution(x_fine, t_val), 'k-', linewidth=2, label='Точное решение')
    plt.plot(x_num, u_ftcs, 'bo--', markersize=4, label='Явная схема FTCS')
    plt.plot(x_num, u_ie, 'rs:', markersize=4, label='Неявная схема Эйлера')
    plt.title(f'Профиль температуры при t = {t_val} (N = {N_plot}, r = 0.4)')
    plt.xlabel('x')
    plt.ylabel('u(x, t)')
    plt.grid(True)
    plt.legend()
    plt.show()

### Шаг 3. Таблица ошибок и порядок сходимости

Вычислим максимальную абсолютную ошибку $E_\infty = \max_j |u_j - u(x_j, t)|$ на сетках $N = 25, 50, 100$ при $r = 0.4$ и оценим порядок сходимости:
$$p = \log_2 \frac{E_\infty(N/2)}{E_\infty(N)}$$

In [ ]:
N_list = [25, 50, 100]

for method_name, solver in [("Явная схема FTCS", solve_ftcs), ("Неявная схема Эйлера", solve_implicit_euler)]:
    print(f"=== {method_name} (r = 0.4) ===")
    print(f"{'t':>7} | {'N':>5} | {'dx':>8} | {'dt':>10} | {'E_inf':>12} | {'Порядок p':>10}")
    print("-" * 66)
    for t_val in t_list:
        errors = []
        for idx, N in enumerate(N_list):
            dx = 1.0 / N
            dt = 0.4 * dx**2
            x_num, u_num = solver(N, t_val, r=0.4)
            err = np.max(np.abs(u_num - exact_solution(x_num, t_val)))
            errors.append(err)
            if idx == 0:
                p_str = "-"
            else:
                p_val = np.log2(errors[idx - 1] / errors[idx])
                p_str = f"{p_val:.2f}"
            print(f"{t_val:7.3f} | {N:5d} | {dx:8.4f} | {dt:10.6f} | {err:12.6e} | {p_str:>10}")
        print("-" * 66)
    print()

---

## Часть 2. Схема Кранка—Николсона и нагретый участок

### Шаг 4. Реализация схемы Кранка—Николсона

Начальный профиль заменяется ступенькой:
$$u(x, 0) = \begin{cases} 1, & 0.4 \le x \le 0.6, \\ 0, & \text{иначе.} \end{cases}$$

Схема Кранка—Николсона использует полусумму вторых разностей на слоях $n$ и $n+1$:
$$-\frac{r}{2}u_{j-1}^{n+1} + (1 + r)u_j^{n+1} - \frac{r}{2}u_{j+1}^{n+1} = \frac{r}{2}u_{j-1}^n + (1 - r)u_j^n + \frac{r}{2}u_{j+1}^n$$

In [ ]:
# Начальный профиль с нагретым участком
def u0_step(x):
    u = np.zeros_like(x)
    for j in range(len(x)):
        if 0.4 - 1e-12 <= x[j] <= 0.6 + 1e-12:
            u[j] = 1.0
    return u

# Один шаг явной схемы FTCS
def step_ftcs(u, r):
    N = len(u) - 1
    u_next = np.zeros_like(u)
    for j in range(1, N):
        u_next[j] = u[j] + r * (u[j - 1] - 2.0 * u[j] + u[j + 1])
    return u_next

# Один шаг неявной схемы Эйлера
def step_implicit_euler(u, r):
    m = len(u) - 2
    a = np.full(m, -r)
    b = np.full(m, 1.0 + 2.0 * r)
    c = np.full(m, -r)
    d = u[1:-1].copy()
    u_next = np.zeros_like(u)
    u_next[1:-1] = thomas_solve(a, b, c, d)
    return u_next

# Один шаг схемы Кранка-Николсона
def step_crank_nicolson(u, r):
    m = len(u) - 2
    a = np.full(m, -0.5 * r)
    b = np.full(m, 1.0 + r)
    c = np.full(m, -0.5 * r)
    d = np.zeros(m)
    for i in range(m):
        j = i + 1
        d[i] = 0.5 * r * u[j - 1] + (1.0 - r) * u[j] + 0.5 * r * u[j + 1]
    u_next = np.zeros_like(u)
    u_next[1:-1] = thomas_solve(a, b, c, d)
    return u_next

### Шаг 5. Первые временные слои при $r = 0.4, 2, 10$

На одной сетке $N = 25$ ($\Delta x = 0.04$) построим первые временные слои для $r = 0.4, 2, 10$ и отдельно покажем увеличенные фрагменты около обеих границ нагретого участка $x = 0.4$ и $x = 0.6$. Явная схема FTCS при $r > 0.5$ неустойчива, поэтому большие шаги применяются только к неявным схемам.

In [ ]:
N_grid = 25
x_grid = np.linspace(0.0, 1.0, N_grid + 1)
r_values = [0.4, 2.0, 10.0]

for r_val in r_values:
    u_ie_1 = step_implicit_euler(u0_step(x_grid), r_val)
    u_ie_2 = step_implicit_euler(u_ie_1, r_val)

    u_cn_1 = step_crank_nicolson(u0_step(x_grid), r_val)
    u_cn_2 = step_crank_nicolson(u_cn_1, r_val)

    # Профиль на первых слоях
    plt.figure(figsize=(7, 5))
    plt.plot(x_grid, u0_step(x_grid), 'k--', label='Слой 0')
    if r_val <= 0.5:
        u_ex_1 = step_ftcs(u0_step(x_grid), r_val)
        plt.plot(x_grid, u_ex_1, 'g-', label='FTCS (слой 1)')
    plt.plot(x_grid, u_ie_1, 'b-', label='Неявный Эйлер (слой 1)')
    plt.plot(x_grid, u_cn_1, 'r-o', markersize=3, label='Кранк-Николсон (слой 1)')
    plt.plot(x_grid, u_cn_2, 'm--s', markersize=3, label='Кранк-Николсон (слой 2)')
    plt.title(f'Первые временные слои при r = {r_val} (N = {N_grid})')
    plt.xlabel('x')
    plt.ylabel('u(x, t)')
    plt.grid(True)
    plt.legend()
    plt.show()

    # Увеличенные фрагменты около обеих границ нагретого участка: x = 0.4 и x = 0.6
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, x_b, (lo, hi) in zip(axes, [0.4, 0.6], [(0.2, 0.6), (0.4, 0.8)]):
        mask = (x_grid >= lo - 1e-12) & (x_grid <= hi + 1e-12)
        ax.plot(x_grid[mask], u0_step(x_grid)[mask], 'k--', label='Слой 0')
        if r_val <= 0.5:
            ax.plot(x_grid[mask], u_ex_1[mask], 'g-^', markersize=4, label='FTCS (слой 1)')
        ax.plot(x_grid[mask], u_ie_1[mask], 'b-o', markersize=4, label='Неявный Эйлер (слой 1)')
        ax.plot(x_grid[mask], u_cn_1[mask], 'r-o', markersize=4, label='Кранк-Николсон (слой 1)')
        ax.plot(x_grid[mask], u_cn_2[mask], 'm--s', markersize=4, label='Кранк-Николсон (слой 2)')
        ax.axvline(x_b, color='gray', linewidth=0.8)
        ax.set_title(f'Около границы x = {x_b} при r = {r_val}')
        ax.set_xlabel('x')
        ax.set_ylabel('u(x, t)')
        ax.grid(True)
        ax.legend()
    plt.show()

### Шаг 6. Минимум и максимум решения и сопоставление с малым шагом

Все расчёты шага 5 и шага 6 выполняются на одной пространственной сетке $N = 25$ ($\Delta x = 0.04$) при $r = 0.4, 2, 10$. Сопоставим профили неявных схем в момент времени $t = 0.064$ (его можно набрать целым числом шагов при каждом из $r$) с расчётом на малом временном шаге ($r = 0.05$, схема Кранка—Николсона). Затем построим графики минимума и максимума решения во времени для каждого $r$ и выведем таблицу экстремумов. Отрицательные значения не обрезаются: на этой сетке у схемы Кранка—Николсона при $r = 10$ температура выходит в отрицательную область.

In [ ]:
# Сопоставление с расчетом на малом шаге при t = 0.064 (N = 25)
t_compare = 0.064
dx = 1.0 / N_grid

r_small = 0.05
dt_small = r_small * dx**2
n_small = int(round(t_compare / dt_small))
u_ref = u0_step(x_grid)
for _ in range(n_small):
    u_ref = step_crank_nicolson(u_ref, r_small)

plt.figure(figsize=(8, 5))
plt.plot(x_grid, u_ref, 'k-', linewidth=2, label='Малый шаг (r = 0.05)')

dev_rows = []
for r_val, style_ie, style_cn in [(0.4, 'b--', 'r--'), (2.0, 'c-.', 'm-.'), (10.0, 'g:', 'y-o')]:
    dt_val = r_val * dx**2
    n_steps = int(round(t_compare / dt_val))
    assert abs(n_steps * dt_val - t_compare) < 1e-12
    u_ie = u0_step(x_grid)
    u_cn = u0_step(x_grid)
    for _ in range(n_steps):
        u_ie = step_implicit_euler(u_ie, r_val)
        u_cn = step_crank_nicolson(u_cn, r_val)
    plt.plot(x_grid, u_ie, style_ie, label=f'Неявный Эйлер (r = {r_val})')
    plt.plot(x_grid, u_cn, style_cn, markersize=3, label=f'Кранк-Николсон (r = {r_val})')
    dev_rows.append((r_val, np.max(np.abs(u_ie - u_ref)), np.max(np.abs(u_cn - u_ref))))

plt.title(f'Сопоставление с малым шагом при t = {t_compare} (N = {N_grid})')
plt.xlabel('x')
plt.ylabel('u(x, t)')
plt.grid(True)
plt.legend()
plt.show()

print(f"Максимальное отклонение от расчета на малом шаге при t = {t_compare} (N = {N_grid}):")
print(f"{'r':>5} | {'Неявный Эйлер':>15} | {'Кранк-Николсон':>15}")
print("-" * 42)
for r_val, d_ie, d_cn in dev_rows:
    print(f"{r_val:5.1f} | {d_ie:15.6e} | {d_cn:15.6e}")
print()

# Динамика минимума и максимума во времени на той же сетке N = 25
t_max_sim = 0.32

print(f"Экстремумы решения (N = {N_grid}, t <= {t_max_sim})")
print(f"{'Метод':>18} | {'r':>5} | {'Слой 1: min':>12} | {'Слой 1: max':>12} | {'min за все время':>17} | {'max за все время':>17}")
print("-" * 98)

for r_val in r_values:
    dt_c = r_val * dx**2
    steps_c = int(round(t_max_sim / dt_c))
    assert abs(steps_c * dt_c - t_max_sim) < 1e-12

    u_ie = u0_step(x_grid)
    u_cn = u0_step(x_grid)

    t_hist = [0.0]
    min_ie, max_ie = [u_ie.min()], [u_ie.max()]
    min_cn, max_cn = [u_cn.min()], [u_cn.max()]

    for s in range(1, steps_c + 1):
        u_ie = step_implicit_euler(u_ie, r_val)
        u_cn = step_crank_nicolson(u_cn, r_val)
        t_hist.append(s * dt_c)
        min_ie.append(u_ie.min())
        max_ie.append(u_ie.max())
        min_cn.append(u_cn.min())
        max_cn.append(u_cn.max())

    print(f"{'Неявный Эйлер':>18} | {r_val:5.1f} | {min_ie[1]:12.6f} | {max_ie[1]:12.6f} | {min(min_ie):17.6e} | {max(max_ie):17.6e}")
    print(f"{'Кранк-Николсон':>18} | {r_val:5.1f} | {min_cn[1]:12.6f} | {max_cn[1]:12.6f} | {min(min_cn):17.6e} | {max(max_cn):17.6e}")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].plot(t_hist, max_ie, 'b-', label='max (Неявный Эйлер)')
    axes[0].plot(t_hist, max_cn, 'r--', label='max (Кранк-Николсон)')
    axes[0].set_title(f'Максимум решения во времени (N = {N_grid}, r = {r_val})')
    axes[0].set_ylabel('max u(x, t)')
    axes[1].plot(t_hist, min_ie, 'b-', label='min (Неявный Эйлер)')
    axes[1].plot(t_hist, min_cn, 'r-o', markersize=3, label='min (Кранк-Николсон)')
    axes[1].axhline(0.0, color='gray', linewidth=0.8)
    axes[1].set_title(f'Минимум решения во времени (N = {N_grid}, r = {r_val})')
    axes[1].set_ylabel('min u(x, t)')
    for ax in axes:
        ax.set_xlabel('t')
        ax.grid(True)
        ax.legend()
    plt.show()

---

## Выводы

1. **Порядок сходимости и сравнение FTCS и неявного Эйлера:**
   * При фиксированном $r = \Delta t / \Delta x^2 = 0.4$ шаг $\Delta t$ убывает пропорционально $\Delta x^2$. Поэтому обе схемы показывают второй порядок сходимости ($p \approx 2.0$) при сгущении сетки.
   * Ошибка явной схемы FTCS примерно в 2.4 раза меньше ошибки неявной схемы Эйлера, так как погрешности аппроксимации по времени и по пространству в FTCS имеют разные знаки и частично сокращаются.

2. **Почему мелкие детали температурного профиля исчезают быстрее крупных?**
   * В точном решении гармоника $\sin(k\pi x)$ затухает с множителем $e^{-k^2\pi^2 t}$. Для мелких деталей ($k = 5$) показатель затухания равен $25\pi^2$, что в 25 раз больше показателя основной гармоники ($k = 1$, $\pi^2$). Из-за больших локальных градиентов и кривизны $|u_{xx}|$ тепловой поток быстро сглаживает частые колебания профиля.

3. **Гарантирует ли безусловная устойчивость точность и отсутствие осцилляций?**
   * **Нет, не гарантирует.** Безусловная устойчивость лишь не даёт решению расти до бесконечности.
   * Схема Кранка—Николсона при больших шагах ($r = 2, 10$) на разрывном начальном профиле даёт нефизические осцилляции вблизи границ нагретого участка ($x = 0.4$ и $x = 0.6$). При $r = 2$ значения остаются неотрицательными, а при $r = 10$ на сетке $N = 25$ минимум решения уходит в отрицательную область ($\min u \approx -0.0047$). Причина в том, что при $r > 1$ коэффициент $(1 - r)$ в правой части схемы отрицателен, и схема перестаёт быть монотонной. Кроме того, множитель перехода для самых высоких гармоник $G(\pi) = \frac{1 - 2r}{1 + 2r}$ отрицателен уже при $r > 0.5$ и стремится к $-1$ при $r \to \infty$: такие гармоники почти не затухают и меняют знак на каждом шаге (нет $L$-устойчивости).
   * Неявная схема Эйлера монотонна и $L$-устойчива (осцилляций и отрицательных температур нет при любых $r$), но при больших $r = 2, 10$ заметно размазывает профиль по сравнению с расчётом на малом шаге из-за первого порядка точности по времени.
   * Сопоставление с расчётом на малом шаге ($r = 0.05$, $t = 0.064$) показывает, что при $r = 0.4$ и $r = 2$ Кранк—Николсон намного точнее неявного Эйлера (максимальное отклонение $\sim 10^{-6}$ и $6 \cdot 10^{-5}$ против $7 \cdot 10^{-4}$ и $3.7 \cdot 10^{-3}$). При $r = 10$ ситуация обратная: отклонение Кранка—Николсона ($\approx 0.19$) примерно в 8 раз больше, чем у неявного Эйлера ($\approx 0.022$). Устойчивость сохраняется, но точность и монотонность теряются.